In [ ]:
import os

import pandas as pd
import plotly.express as px

from util import CensusApi, compute_headship_rates, YEAR_CONFIG

c = CensusApi(os.getenv("CENSUS_KEY"), timeout=90)

county_ids = [53033, 53035, 53053, 53061]
state_id = 53


def fetch_totals(cols_dict, year, dataset):
    """Pull a decennial table and return a DataFrame indexed by county geoid
    with one column per age group."""
    df = c.get_dec_data(cols_dict, year, 'county', dataset, county_ids, state_id)
    df = df.loc[df.geoid.isin(county_ids)].drop(columns=['name']).set_index('geoid')
    return df


In [ ]:
headship_rates = compute_headship_rates(YEAR_CONFIG, fetch_fn=fetch_totals)

# Add a 0 headship rate for the under-15 age group for each county so the
# index aligns with the PUMS hhpop series (which includes 0-14).
zeros = pd.DataFrame(
    0.0,
    index=pd.MultiIndex.from_product(
        [county_ids, ['age_0_14']], names=headship_rates.index.names
    ),
    columns=headship_rates.columns,
)
headship_rates = pd.concat([headship_rates, zeros]).sort_index()
headship_rates


headship_rate_2000  headship_rate_2010  \
county_id age                                                   
53033     age_0_14               0.000000            0.000000   
          age_15_24              0.190041            0.163842   
          age_25_34              0.504403            0.485326   
          age_35_44              0.557491            0.552637   
          age_45_54              0.594812            0.582187   
          age_55_64              0.619121            0.605494   
          age_65_74              0.625346            0.631974   
          age_75_84              0.696796            0.682283   
          age_85_plus            0.768192            0.754121   
53035     age_0_14               0.000000            0.000000   
          age_15_24              0.167348            0.149028   
          age_25_34              0.488975            0.456558   
          age_35_44              0.537136            0.526911   
          age_45_54              0.567255            0.559850   
          age_55_64              0.609939            0.582795   
          age_65_74              0.628685            0.634481   
          age_75_84              0.698451            0.693312   
          age_85_plus            0.733578            0.759316   
53053     age_0_14               0.000000            0.000000   
          age_15_24              0.169159            0.143947   
          age_25_34              0.497748            0.470108   
          age_35_44              0.544255            0.537777   
          age_45_54              0.584725            0.568756   
          age_55_64              0.607257            0.590965   
          age_65_74              0.639744            0.629875   
          age_75_84              0.696703            0.693466   
          age_85_plus            0.786235            0.749371   
53061     age_0_14               0.000000            0.000000   
          age_15_24              0.145350            0.113242   
          age_25_34              0.479013            0.442442   
          age_35_44              0.547208            0.528749   
          age_45_54              0.577424            0.570118   
          age_55_64              0.600431            0.590427   
          age_65_74              0.616345            0.623945   
          age_75_84              0.691334            0.676047   
          age_85_plus            0.705695            0.729472   

                       headship_rate_2020  
county_id age                              
53033     age_0_14               0.000000  
          age_15_24              0.156006  
          age_25_34              0.484709  
          age_35_44              0.537512  
          age_45_54              0.562143  
          age_55_64              0.578647  
          age_65_74              0.606555  
          age_75_84              0.634950  
          age_85_plus            0.710742  
53035     age_0_14               0.000000  
          age_15_24              0.131984  
          age_25_34              0.419648  
          age_35_44              0.497553  
          age_45_54              0.531000  
          age_55_64              0.568454  
          age_65_74              0.603145  
          age_75_84              0.647476  
          age_85_plus            0.713971  
53053     age_0_14               0.000000  
          age_15_24              0.122025  
          age_25_34              0.434400  
          age_35_44              0.511378  
          age_45_54              0.543645  
          age_55_64              0.567371  
          age_65_74              0.602443  
          age_75_84              0.637337  
          age_85_plus            0.699887  
53061     age_0_14               0.000000  
          age_15_24              0.096906  
          age_25_34              0.407167  
          age_35_44              0.507356  
          age_45_54              0.538187  
          age_55_64              0.568929  
          ag

In [21]:
pums = pd.read_csv('data/pums2023_hhpop.csv')
PUMS_AGE_MAP = {
    'ages_0_4': 'age_0_14', 'ages_5_9': 'age_0_14', 'ages_10_14': 'age_0_14',
    'ages_15_19': 'age_15_24', 'ages_20_24': 'age_15_24',
    'ages_25_29': 'age_25_34', 'ages_30_34': 'age_25_34',
    'ages_35_39': 'age_35_44', 'ages_40_44': 'age_35_44',
    'ages_45_49': 'age_45_54', 'ages_50_54': 'age_45_54',
    'ages_55_59': 'age_55_64', 'ages_60_64': 'age_55_64',
    'ages_65_69': 'age_65_74', 'ages_70_74': 'age_65_74',
    'ages_75_79': 'age_75_84', 'ages_80_84': 'age_75_84',
    'ages_85_plus': 'age_85_plus',
}

pums = (
    pums.assign(age=pums['age'].map(PUMS_AGE_MAP))
    .dropna(subset=['age'])
    .groupby(['county_id','age'])['hhpop'].sum()
)

In [22]:
df = headship_rates.copy()
df['hhpop'] = pums
df['hh_2000'] = df['headship_rate_2000'] * df['hhpop']
df['hh_2010'] = df['headship_rate_2010'] * df['hhpop']
df['hh_2020'] = df['headship_rate_2020'] * df['hhpop']

In [23]:
df.reset_index().to_csv('data/headship_rates_by_age.csv', index=False)

In [24]:
remi = pd.read_csv('data/remi2060_hhpop.csv')
remi = (
    remi.assign(age=remi['age'].map(PUMS_AGE_MAP))
    .dropna(subset=['age'])
    .groupby(['county_id','age'])['hhpop'].sum()
)

In [25]:
df = headship_rates.copy()
df['hhpop'] = remi
df['hh_2000'] = df['headship_rate_2000'] * df['hhpop']
df['hh_2010'] = df['headship_rate_2010'] * df['hhpop']
df['hh_2020'] = df['headship_rate_2020'] * df['hhpop']

In [26]:
df.reset_index().to_csv('data/remi2060_headship_rates_by_age.csv', index=False)